- 001 hello world (should integrate the encoding step into this file)

In [ ]:
from ppp_paths import dataint


In [ ]:
import pandas as pd
import json, requests 
import networkx as nx
import gc
# import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from tqdm import tqdm
import gc
import time
import sys
import unicodedata
from metaphone import doublemetaphone
from fuzzywuzzy import fuzz
# from difflib import SequenceMatcher
import re
from math import radians, cos, sin, asin, sqrt
import datetime 
from datetime import date

# window = (365*5)


In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')

In [ ]:
paperdataembed = pd.read_parquet(dataint + 'embeddedpapers.parquet')
gc.collect()
paperdataembed = paperdataembed.drop_duplicates(subset=['id'], keep='first')
paperdataembed = paperdataembed.rename(columns={'id': 'magid'})
paperdataembed['magid'] = paperdataembed['magid'].astype(str)
paperdataembed = paperdataembed.rename(columns={'encoded_title': 'encoded_paper_title'})  
paperdataembed = paperdataembed.rename(columns={'encoded_abstract': 'encoded_paper_abstract'})  
print(paperdataembed.shape)
paperdataembed


In [ ]:
tempy = paperdataembed.copy()
tempy = tempy[tempy['magid']=='15883']
tempy

In [ ]:
possiblepairs = possiblepairs.merge(paperdataembed, on='magid', how='left')
possiblepairs

In [ ]:
patentdataembed = pd.read_parquet(dataint + 'embeddedpatents.parquet')
patentdataembed = patentdataembed.rename(columns={'id': 'patent_id'})
patentdataembed['patent_id'] = patentdataembed['patent_id'].astype(str)
patentdataembed.drop_duplicates(subset=['patent_id'], inplace=True)
patentdataembed = patentdataembed.rename(columns={'encoded_title': 'encoded_patent_title'})  
patentdataembed = patentdataembed.rename(columns={'encoded_abstract': 'encoded_patent_abstract'})  

print(patentdataembed.shape)
# pppappdataembed


In [ ]:
tempy = patentdataembed.copy()
tempy = tempy[tempy['patent_id']=='10824716']
tempy

In [ ]:
possiblepairs = possiblepairs.merge(patentdataembed, on='patent_id', how='left')
# del patentdataembed
gc.collect()
possiblepairs

In [ ]:
tempy = possiblepairs.copy()
tempy = tempy[tempy['magid']=='15883']
tempy = tempy[tempy['patent_id']=='10824716']
tempy

In [ ]:
# print("Number of duplicate rows:", potentialpairs.duplicated().sum())
print("Missing values per column:\n", possiblepairs.isna().sum())
possiblepairs.dropna(inplace=True)
# potentialpairs.drop_duplicates(inplace=True)
possiblepairs=possiblepairs[['magid', 'patent_id', 'encoded_paper_title', 'encoded_paper_abstract', 'encoded_patent_title', 'encoded_patent_abstract']]
possiblepairs


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Configurable batch size (adjust based on VRAM and data size)
BATCH_SIZE = 1000000  # ~1.23GB for 4 tensors with D=768, adjust as needed

# Assume pppapppapersembed_window is the input DataFrame
# Get total number of samples
num_samples = len(possiblepairs)
print(f"Total samples: {num_samples}")

# Initialize columns for results
possiblepairs['title_similarity'] = np.nan
possiblepairs['abstract_similarity'] = np.nan

# Process data in batches
for start_idx in range(0, num_samples, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)
    print(f"Processing batch: {start_idx} to {end_idx}")
    
    # Extract batch data
    batch_indices = range(start_idx, end_idx)
    batch_patent_title = torch.tensor(
        np.stack(possiblepairs['encoded_patent_title'].iloc[batch_indices].values),
        dtype=torch.float32
    ).to(device)
    batch_paper_title = torch.tensor(
        np.stack(possiblepairs['encoded_paper_title'].iloc[batch_indices].values),
        dtype=torch.float32
    ).to(device)
    batch_patent_abstract = torch.tensor(
        np.stack(possiblepairs['encoded_patent_abstract'].values[batch_indices]),
        dtype=torch.float32
    ).to(device)
    batch_paper_abstract = torch.tensor(
        np.stack(possiblepairs['encoded_paper_abstract'].values[batch_indices]),
        dtype=torch.float32
    ).to(device)
    
    # Compute cosine similarities
    title_similarities = F.cosine_similarity(batch_patent_title, batch_paper_title, dim=1)
    abstract_similarities = F.cosine_similarity(batch_patent_abstract, batch_paper_abstract, dim=1)
    
    # Compute norms and handle zero-norm cases
    title_norms = torch.norm(batch_patent_title, dim=1) * torch.norm(batch_paper_title, dim=1)
    abstract_norms = torch.norm(batch_patent_abstract, dim=1) * torch.norm(batch_paper_abstract, dim=1)
    title_similarities = torch.where(title_norms == 0, torch.tensor(float('nan'), device=device), title_similarities)
    abstract_similarities = torch.where(abstract_norms == 0, torch.tensor(float('nan'), device=device), abstract_similarities)
    
    # Store results in DataFrame
    possiblepairs.iloc[batch_indices, possiblepairs.columns.get_loc('title_similarity')] = (
        title_similarities.cpu().numpy()
    )
    possiblepairs.iloc[batch_indices, possiblepairs.columns.get_loc('abstract_similarity')] = (
        abstract_similarities.cpu().numpy()
    )
    
    # Clear GPU memory
    del batch_patent_title, batch_paper_title, batch_patent_abstract, batch_paper_abstract
    del title_similarities, abstract_similarities, title_norms, abstract_norms
    torch.cuda.empty_cache()

print("\nFinal DataFrame:")
print(possiblepairs[['magid', 'patent_id', 'title_similarity', 'abstract_similarity']])

In [ ]:
columns_to_drop = ['encoded_patent_title', 'encoded_patent_abstract', 
                   'encoded_paper_title', 'encoded_paper_abstract']
possiblepairs = possiblepairs.drop(columns=columns_to_drop, errors='ignore')
possiblepairs
possiblepairs = possiblepairs[['magid', 'patent_id', 'title_similarity', 'abstract_similarity']]
possiblepairs

In [ ]:
possiblepairs.to_parquet(dataint + 'possiblepairs_contentsimliarity.parquet', index=False, compression='snappy')
